# Examen Aplicado — Machine Learning I
## Predicción del valor mediano de viviendas en California
**Autor:** Rodrigo Soto — Universidad Mayor, Escuela de Ingeniería

### Declaración del dataset

| Campo | Valor |
|---|---|
| **Nombre del dataset** | California Housing Prices (censo de 1990) |
| **Fuente** | Kaggle |
| **URL** | https://www.kaggle.com/datasets/camnugent/california-housing-prices |
| **Licencia** | CC0 — Dominio público |
| **N° de filas** | 20.640 |
| **N° de columnas** | 10 (9 variables predictoras + 1 variable objetivo) |
| **Variable objetivo (y)** | `median_house_value` — valor mediano de la vivienda en el distrito (USD) |
| **Tipo de tarea** | **Regresión** |

El dataset contiene información agregada a nivel de distrito censal (block group) del
censo de California de 1990: ubicación geográfica, antigüedad de las viviendas, número
total de habitaciones y dormitorios, población, número de hogares, ingreso mediano de los
residentes y proximidad al océano. Cumple los requisitos del examen: más de 500
observaciones (20.640), más de 6 variables predictoras (9), de las cuales 8 son numéricas
continuas, y URL pública verificable.

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
os.makedirs("figures", exist_ok=True)
RESULTS = {}
print("Entorno listo. Semilla fija =", SEED)

Entorno listo. Semilla fija = 42


## 1. Carga y exploración inicial

In [2]:
df = pd.read_csv("data/housing.csv")
print("Dimensiones (filas, columnas):", df.shape)

Dimensiones (filas, columnas): (20640, 10)


In [3]:
print("Tipos de datos por columna:")
print(df.dtypes)

Tipos de datos por columna:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity           str
dtype: object


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [5]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [6]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


El dataset tiene 20.640 filas y 10 columnas. Ocho variables son numéricas continuas
(`longitude`, `latitude`, `housing_median_age`, `total_rooms`, `total_bedrooms`,
`population`, `households`, `median_income`), una es categórica nominal
(`ocean_proximity`) y `median_house_value` es la variable objetivo.

## 2. Análisis de valores faltantes

In [7]:
missing_pct = (df.isna().mean() * 100).round(4).sort_values(ascending=False)
missing_table = pd.DataFrame({"pct_nulos": missing_pct, "n_nulos": df.isna().sum()})
print(missing_table)
RESULTS["missing"] = missing_table.to_dict()

plt.figure(figsize=(10, 5))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Heatmap de valores faltantes por columna")
plt.xlabel("Variables"); plt.ylabel("Observaciones")
plt.tight_layout(); plt.savefig("figures/01_missing_heatmap.png", dpi=150); plt.close()

                    pct_nulos  n_nulos
households             0.0000        0
housing_median_age     0.0000        0
latitude               0.0000        0
longitude              0.0000        0
median_house_value     0.0000        0
median_income          0.0000        0
ocean_proximity        0.0000        0
population             0.0000        0
total_bedrooms         1.0029      207
total_rooms            0.0000        0


**Decisión sobre valores faltantes:** la única columna con nulos es `total_bedrooms`
(207 registros, 1,00% del total). Dado que el porcentaje es muy bajo (<5%) y que se trata
de una variable numérica continua con distribución asimétrica a la derecha, se **imputa con
la mediana** en lugar de eliminar las filas: eliminarlas descartaría 207 distritos
completos con información válida en las otras 9 variables, mientras que la imputación por
mediana es robusta ante los valores extremos presentes en esta variable. La imputación se
realiza **dentro del ColumnTransformer**, ajustada sólo con los datos de entrenamiento,
para evitar data leakage.

## 3. Detección y tratamiento de outliers (método IQR)

In [8]:
NUM_KEY = ["median_income", "total_rooms", "population", "households", "total_bedrooms", "housing_median_age"]

def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

outlier_report = []
for c in NUM_KEY:
    lo, hi = iqr_bounds(df[c])
    n_out = int(((df[c] < lo) | (df[c] > hi)).sum())
    outlier_report.append({"variable": c, "limite_inf": round(lo, 2), "limite_sup": round(hi, 2),
                            "n_outliers": n_out, "pct_outliers": round(n_out / len(df) * 100, 2)})
outlier_df = pd.DataFrame(outlier_report)
print(outlier_df)
RESULTS["outliers"] = outlier_df.to_dict(orient="records")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.flat, NUM_KEY):
    sns.boxplot(y=df[c], ax=ax, color="#DD8452")
    ax.set_title(f"{c} (antes)"); ax.set_ylabel(c)
plt.suptitle("Boxplots ANTES del tratamiento de outliers (método IQR)")
plt.tight_layout(); plt.savefig("figures/02_boxplots_antes.png", dpi=150); plt.close()

# Tratamiento: winsorización (capping) a los límites IQR, no eliminación
df_treated = df.copy()
for c in NUM_KEY:
    lo, hi = iqr_bounds(df[c])
    df_treated[c] = df_treated[c].clip(lower=lo, upper=hi)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.flat, NUM_KEY):
    sns.boxplot(y=df_treated[c], ax=ax, color="#55A868")
    ax.set_title(f"{c} (después)"); ax.set_ylabel(c)
plt.suptitle("Boxplots DESPUÉS del tratamiento de outliers (winsorización IQR)")
plt.tight_layout(); plt.savefig("figures/03_boxplots_despues.png", dpi=150); plt.close()

             variable  limite_inf  limite_sup  n_outliers  pct_outliers
0       median_income       -0.71        8.01         681          3.30
1         total_rooms    -1102.62     5698.38        1287          6.24
2          population     -620.00     3132.00        1196          5.79
3          households     -207.50     1092.50        1220          5.91
4      total_bedrooms     -230.50     1173.50        1271          6.16
5  housing_median_age      -10.50       65.50           0          0.00


**Decisión sobre outliers:** se aplica **winsorización (capping)** a los límites del rango
intercuartílico en lugar de eliminar las observaciones. La justificación es doble: (a) los
valores extremos de `total_rooms`, `population` y `households` corresponden a distritos
censales genuinamente grandes, no a errores de medición — eliminarlos sesgaría el modelo
hacia distritos pequeños y perdería entre 4% y 6% de los datos por variable; (b) el capping
preserva el tamaño muestral completo y reduce la influencia desproporcionada de las colas
sobre los modelos lineales, que son sensibles a valores extremos. Se conserva `median_income`
acotado porque su cola derecha (ingresos muy altos) es informativa pero domina la escala.

## 4. Distribución de la variable objetivo

In [9]:
skew_target = df["median_house_value"].skew()
print(f"Skewness de median_house_value: {skew_target:.4f}")
RESULTS["skew_target"] = round(float(skew_target), 4)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df["median_house_value"], kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title(f"Distribución de median_house_value (skew={skew_target:.3f})")
axes[0].set_xlabel("Valor mediano de vivienda (USD)"); axes[0].set_ylabel("Frecuencia")
sns.histplot(np.log1p(df["median_house_value"]), kde=True, ax=axes[1], color="#55A868")
axes[1].set_title(f"log1p(median_house_value) (skew={np.log1p(df['median_house_value']).skew():.3f})")
axes[1].set_xlabel("log(valor mediano de vivienda)"); axes[1].set_ylabel("Frecuencia")
plt.tight_layout(); plt.savefig("figures/04_target_distribucion.png", dpi=150); plt.close()

Skewness de median_house_value: 0.9778


La variable objetivo presenta una asimetría de aproximadamente 0,98, es decir **por debajo
del umbral de 1,0**, por lo que **no se aplica transformación logarítmica**: la distribución
es moderadamente asimétrica a la derecha pero no lo suficiente como para justificar
transformar la escala, y mantener la escala original permite interpretar directamente las
métricas (RMSE y MAE en dólares). Se observa además el conocido truncamiento del dataset en
USD 500.001: los distritos con valores superiores fueron censurados en ese tope, lo que
genera el pico artificial en el extremo derecho del histograma y constituye una limitación
estructural del dataset que se discute en las conclusiones.

## 5. Visualizaciones exploratorias

In [10]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr(method="pearson")

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Heatmap de correlación de Pearson entre variables numéricas")
plt.xlabel("Variables"); plt.ylabel("Variables")
plt.tight_layout(); plt.savefig("figures/05_corr_heatmap.png", dpi=150); plt.close()

target_corr = corr["median_house_value"].drop("median_house_value").abs().sort_values(ascending=False)
print("Top 5 variables más correlacionadas con median_house_value (|r|):")
print(target_corr.head(5).round(4))
RESULTS["top5_corr_target"] = target_corr.head(5).round(4).to_dict()

top2 = target_corr.head(2).index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, c in zip(axes, top2):
    ax.scatter(df[c], df["median_house_value"], alpha=0.15, s=8, color="#4C72B0")
    ax.set_title(f"{c} vs median_house_value (r={corr.loc[c,'median_house_value']:.3f})")
    ax.set_xlabel(c); ax.set_ylabel("median_house_value (USD)")
plt.tight_layout(); plt.savefig("figures/06_scatter_top2.png", dpi=150); plt.close()

plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x="ocean_proximity", y="median_house_value", palette="Set2")
plt.title("Distribución del valor de vivienda según proximidad al océano")
plt.xlabel("Proximidad al océano"); plt.ylabel("median_house_value (USD)")
plt.tight_layout(); plt.savefig("figures/07_violin_ocean.png", dpi=150); plt.close()

# Multicolinealidad
high_corr = []
cols_c = corr.columns.drop("median_house_value")
for i in range(len(cols_c)):
    for j in range(i + 1, len(cols_c)):
        r = corr.loc[cols_c[i], cols_c[j]]
        if abs(r) > 0.85:
            high_corr.append((cols_c[i], cols_c[j], round(float(r), 4)))
print("Pares de predictores con |r| > 0.85 (multicolinealidad):", high_corr)
RESULTS["multicolinealidad"] = high_corr

Top 5 variables más correlacionadas con median_house_value (|r|):
median_income         0.6881
latitude              0.1442
total_rooms           0.1342
housing_median_age    0.1056
households            0.0658
Name: median_house_value, dtype: float64


Pares de predictores con |r| > 0.85 (multicolinealidad): [('longitude', 'latitude', -0.9247), ('total_rooms', 'total_bedrooms', 0.9304), ('total_rooms', 'population', 0.8571), ('total_rooms', 'households', 0.9185), ('total_bedrooms', 'population', 0.8777), ('total_bedrooms', 'households', 0.9797), ('population', 'households', 0.9072)]


## 6. División train/test (ANTES de cualquier transformación)

In [11]:
# Feature engineering: variable ORDINAL derivada de la antigüedad de las viviendas
def age_bucket(x):
    if x <= 15: return "Nueva"
    elif x <= 30: return "Media"
    elif x <= 45: return "Antigua"
    return "Muy antigua"

data = df_treated.copy()
data["age_category"] = data["housing_median_age"].apply(age_bucket)

X = data.drop(columns=["median_house_value"])
y = data["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
RESULTS["split"] = {"X_train": list(X_train.shape), "X_test": list(X_test.shape)}

X_train: (16512, 10) | X_test: (4128, 10)
y_train: (16512,) | y_test: (4128,)


La división se realiza **antes** de imputar, codificar o escalar. Al tratarse de un problema
de regresión no se utiliza `stratify` (aplicable sólo a clasificación).

## 7. ColumnTransformer (pipeline sin data leakage)

In [12]:
NUM_FEATURES = ["longitude","latitude","housing_median_age","total_rooms","total_bedrooms",
                "population","households","median_income"]
NOM_FEATURES = ["ocean_proximity"]
ORD_FEATURES = ["age_category"]
ORD_CATEGORIES = [["Nueva","Media","Antigua","Muy antigua"]]

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())])
nominal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
ordinal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("ord", OrdinalEncoder(categories=ORD_CATEGORIES,
                                                  handle_unknown="use_encoded_value", unknown_value=-1))])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUM_FEATURES),
    ("nom", nominal_pipe, NOM_FEATURES),
    ("ord", ordinal_pipe, ORD_FEATURES),
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)   # SOLO transform, nunca fit sobre test

feat_names = (NUM_FEATURES
              + list(preprocessor.named_transformers_["nom"]["ohe"].get_feature_names_out(NOM_FEATURES))
              + ORD_FEATURES)
print("Dimensiones tras preprocesamiento -> train:", X_train_prep.shape, "| test:", X_test_prep.shape)
print("Variables generadas:", len(feat_names))

Dimensiones tras preprocesamiento -> train: (16512, 14) | test: (4128, 14)
Variables generadas: 14


## 8. PCA sobre X_train preprocesado

In [13]:
pca_full = PCA(random_state=SEED).fit(X_train_prep)
var_ratio = pca_full.explained_variance_ratio_
var_table = pd.DataFrame({
    "componente": [f"PC{i+1}" for i in range(len(var_ratio))],
    "varianza_explicada": np.round(var_ratio, 4),
    "varianza_acumulada": np.round(np.cumsum(var_ratio), 4),
})
print(var_table)
var_table.to_csv("varianza_pca.csv", index=False)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(var_ratio)+1), np.cumsum(var_ratio), marker="o", color="#4C72B0", label="Varianza acumulada")
plt.bar(range(1, len(var_ratio)+1), var_ratio, alpha=0.4, color="#55A868", label="Varianza por componente")
plt.axhline(0.80, ls="--", color="red", label="80%")
plt.axhline(0.90, ls="--", color="orange", label="90%")
plt.xlabel("Componente principal"); plt.ylabel("Proporción de varianza explicada")
plt.title("Scree plot — varianza explicada y acumulada por componente")
plt.legend(); plt.tight_layout(); plt.savefig("figures/08_scree_plot.png", dpi=150); plt.close()

n_comp = int(np.argmax(np.cumsum(var_ratio) >= 0.80) + 1)
cum_at_n = float(np.cumsum(var_ratio)[n_comp-1])
print(f"n_components seleccionado = {n_comp} (varianza acumulada = {cum_at_n:.4f})")
RESULTS["pca"] = {"n_components": n_comp, "var_acumulada": round(cum_at_n, 4)}

pca = PCA(n_components=n_comp, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_prep)

plt.figure(figsize=(8, 6))
sc = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, cmap="viridis", alpha=0.4, s=8)
plt.colorbar(sc, label="median_house_value (USD)")
plt.xlabel(f"PC1 ({var_ratio[0]*100:.1f}% var.)"); plt.ylabel(f"PC2 ({var_ratio[1]*100:.1f}% var.)")
plt.title("Observaciones de entrenamiento en el espacio PC1 vs PC2")
plt.tight_layout(); plt.savefig("figures/09_pca_scatter.png", dpi=150); plt.close()

loadings = pd.DataFrame(pca.components_[:2].T, index=feat_names, columns=["PC1", "PC2"])
print("Top 5 contribuciones a PC1:"); print(loadings["PC1"].abs().sort_values(ascending=False).head(5).round(4))
print("Top 5 contribuciones a PC2:"); print(loadings["PC2"].abs().sort_values(ascending=False).head(5).round(4))
RESULTS["loadings_pc1"] = loadings["PC1"].abs().sort_values(ascending=False).head(5).round(4).to_dict()
RESULTS["loadings_pc2"] = loadings["PC2"].abs().sort_values(ascending=False).head(5).round(4).to_dict()

   componente  varianza_explicada  varianza_acumulada
0         PC1              0.4300              0.4300
1         PC2              0.2103              0.6403
2         PC3              0.1559              0.7962
3         PC4              0.1108              0.9070
4         PC5              0.0331              0.9400
5         PC6              0.0202              0.9603
6         PC7              0.0156              0.9759
7         PC8              0.0097              0.9856
8         PC9              0.0053              0.9910
9        PC10              0.0049              0.9958
10       PC11              0.0024              0.9983
11       PC12              0.0017              1.0000
12       PC13              0.0000              1.0000
13       PC14              0.0000              1.0000


n_components seleccionado = 4 (varianza acumulada = 0.9070)


Top 5 contribuciones a PC1:
households            0.4702
total_bedrooms        0.4697
total_rooms           0.4651
population            0.4497
housing_median_age    0.2686
Name: PC1, dtype: float64
Top 5 contribuciones a PC2:
latitude                     0.6913
longitude                    0.6797
ocean_proximity_<1H OCEAN    0.1593
ocean_proximity_NEAR BAY     0.0972
ocean_proximity_INLAND       0.0861
Name: PC2, dtype: float64


## 9. K-Means: selección del K óptimo

In [14]:
inertias, silhouettes, ks = [], [], list(range(2, 11))
sample_idx = np.random.RandomState(SEED).choice(len(X_train_pca), size=min(5000, len(X_train_pca)), replace=False)
for k in ks:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X_train_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_train_pca[sample_idx], km.labels_[sample_idx]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(ks, inertias, marker="o", color="#4C72B0")
axes[0].set_title("Método del codo (inercia vs K)"); axes[0].set_xlabel("K (n° de clusters)"); axes[0].set_ylabel("Inercia")
axes[1].plot(ks, silhouettes, marker="o", color="#C44E52")
axes[1].set_title("Silhouette Score vs K"); axes[1].set_xlabel("K (n° de clusters)"); axes[1].set_ylabel("Silhouette Score")
plt.tight_layout(); plt.savefig("figures/10_kmeans_seleccion_k.png", dpi=150); plt.close()

k_opt = ks[int(np.argmax(silhouettes))]
print("Tabla K / inercia / silhouette:")
print(pd.DataFrame({"K": ks, "inercia": np.round(inertias, 1), "silhouette": np.round(silhouettes, 4)}))
print("K óptimo seleccionado:", k_opt)
RESULTS["kmeans"] = {"k_optimo": int(k_opt), "silhouette": round(float(max(silhouettes)), 4)}

Tabla K / inercia / silhouette:
    K  inercia  silhouette
0   2  95929.2      0.3167
1   3  73441.4      0.2925
2   4  64478.4      0.2799
3   5  56550.8      0.2568
4   6  51087.4      0.2575
5   7  46351.4      0.2617
6   8  43255.1      0.2601
7   9  40305.9      0.2599
8  10  37954.4      0.2513
K óptimo seleccionado: 2


In [15]:
kmeans = KMeans(n_clusters=k_opt, random_state=SEED, n_init=10).fit(X_train_pca)
clusters = kmeans.labels_

plt.figure(figsize=(8, 6))
for c in range(k_opt):
    m = clusters == c
    plt.scatter(X_train_pca[m, 0], X_train_pca[m, 1], alpha=0.4, s=8, label=f"Cluster {c}")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"Clusters K-Means (K={k_opt}) en el espacio PCA")
plt.legend(); plt.tight_layout(); plt.savefig("figures/11_kmeans_clusters.png", dpi=150); plt.close()

profile = X_train[NUM_FEATURES].copy()
profile["cluster"] = clusters
profile["median_house_value"] = y_train.values
cluster_profile = profile.groupby("cluster").mean().round(2)
print("Perfil de clusters (media por variable):")
print(cluster_profile)
cluster_profile.to_csv("perfil_clusters.csv")
RESULTS["cluster_profile"] = cluster_profile.to_dict()

Perfil de clusters (media por variable):
         longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
cluster                                                                         
0          -119.33     35.38               20.35      4176.60          863.42   
1          -119.69     35.75               31.93      1751.83          358.65   

         population  households  median_income  median_house_value  
cluster                                                             
0           2245.75      800.78           4.03           216910.22  
1            972.19      336.30           3.72           203279.26  


## 10. Modelos supervisados

In [16]:
# --- Modelo penalizado: Ridge ---
t0 = time.time()
ridge_grid = GridSearchCV(Ridge(random_state=SEED),
                           param_grid={"alpha": [0.001, 0.01, 0.1, 1, 10, 100]},
                           cv=5, scoring="r2", n_jobs=-1)
ridge_grid.fit(X_train_prep, y_train)
t_ridge = time.time() - t0
ridge_best = ridge_grid.best_estimator_
print("Ridge — mejores parámetros:", ridge_grid.best_params_)
print(f"Ridge — mejor R² CV (5-fold): {ridge_grid.best_score_:.4f} | tiempo entrenamiento: {t_ridge:.2f}s")

Ridge — mejores parámetros: {'alpha': 1}
Ridge — mejor R² CV (5-fold): 0.6688 | tiempo entrenamiento: 2.07s


In [17]:
# --- Modelo de árboles: Random Forest ---
t0 = time.time()
rf_grid = GridSearchCV(RandomForestRegressor(random_state=SEED, n_jobs=-1),
                        param_grid={"n_estimators": [100, 200, 300],
                                    "max_depth": [10, 20, None],
                                    "min_samples_split": [2, 5, 10]},
                        cv=5, scoring="r2", n_jobs=-1)
rf_grid.fit(X_train_prep, y_train)
t_rf = time.time() - t0
rf_best = rf_grid.best_estimator_
print("Random Forest — mejores parámetros:", rf_grid.best_params_)
print(f"Random Forest — mejor R² CV (5-fold): {rf_grid.best_score_:.4f} | tiempo entrenamiento: {t_rf:.2f}s")
RESULTS["ridge_best"] = ridge_grid.best_params_
RESULTS["rf_best"] = rf_grid.best_params_

Random Forest — mejores parámetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}
Random Forest — mejor R² CV (5-fold): 0.8180 | tiempo entrenamiento: 804.32s


## 11. Métricas sobre el conjunto de test

In [18]:
def eval_model(name, model, t_train):
    t0 = time.time()
    y_pred = model.predict(X_test_prep)
    t_inf = time.time() - t0
    y_pred_train = model.predict(X_train_prep)
    return {
        "modelo": name,
        "RMSE": round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 4),
        "MAE": round(float(mean_absolute_error(y_test, y_pred)), 4),
        "R2": round(float(r2_score(y_test, y_pred)), 4),
        "MAPE": round(float(mean_absolute_percentage_error(y_test, y_pred)), 4),
        "R2_train": round(float(r2_score(y_train, y_pred_train)), 4),
        "tiempo_entrenamiento_s": round(t_train, 4),
        "tiempo_inferencia_s": round(t_inf, 4),
    }, y_pred

res_ridge, pred_ridge = eval_model("Ridge", ridge_best, t_ridge)
res_rf, pred_rf = eval_model("Random Forest", rf_best, t_rf)
for r in (res_ridge, res_rf):
    print(r)

{'modelo': 'Ridge', 'RMSE': 70093.8394, 'MAE': 51452.8302, 'R2': 0.6251, 'MAPE': 0.3013, 'R2_train': 0.6695, 'tiempo_entrenamiento_s': 2.0741, 'tiempo_inferencia_s': 0.0009}
{'modelo': 'Random Forest', 'RMSE': 48901.4813, 'MAE': 31683.3875, 'R2': 0.8175, 'MAPE': 0.1772, 'R2_train': 0.976, 'tiempo_entrenamiento_s': 804.3244, 'tiempo_inferencia_s': 0.2963}


In [19]:
comparison = pd.DataFrame([res_ridge, res_rf]).sort_values("R2", ascending=False).reset_index(drop=True)
comparison.to_csv("tabla_comparativa_modelos.csv", index=False)
print(comparison.to_string(index=False))
RESULTS["comparison"] = comparison.to_dict(orient="records")

best_name = comparison.iloc[0]["modelo"]
best_model = rf_best if best_name == "Random Forest" else ridge_best
best_pred = pred_rf if best_name == "Random Forest" else pred_ridge
RESULTS["best_model"] = best_name

       modelo       RMSE        MAE     R2   MAPE  R2_train  tiempo_entrenamiento_s  tiempo_inferencia_s
Random Forest 48901.4813 31683.3875 0.8175 0.1772    0.9760                804.3244               0.2963
        Ridge 70093.8394 51452.8302 0.6251 0.3013    0.6695                  2.0741               0.0009


## 12. Gráficos de desempeño (regresión)

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, best_pred, alpha=0.2, s=8, color="#4C72B0")
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, "--", color="red")
axes[0].set_xlabel("Valor real (USD)"); axes[0].set_ylabel("Valor predicho (USD)")
axes[0].set_title(f"Valores reales vs. predichos — {best_name}")
residuals = y_test - best_pred
sns.histplot(residuals, kde=True, ax=axes[1], color="#55A868")
axes[1].set_xlabel("Residual (real - predicho, USD)"); axes[1].set_ylabel("Frecuencia")
axes[1].set_title(f"Distribución de residuales — {best_name}")
plt.tight_layout(); plt.savefig("figures/12_real_vs_pred_residuales.png", dpi=150); plt.close()

## 13. Importancia de variables del mejor modelo

In [21]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=feat_names).sort_values(ascending=False)
    titulo = "Importancia de variables (Random Forest)"
else:
    importances = pd.Series(np.abs(best_model.coef_), index=feat_names).sort_values(ascending=False)
    titulo = "Magnitud de coeficientes (Ridge)"

plt.figure(figsize=(9, 7))
importances.head(12).sort_values().plot(kind="barh", color="#55A868")
plt.title(titulo); plt.xlabel("Importancia"); plt.ylabel("Variable")
plt.tight_layout(); plt.savefig("figures/13_feature_importance.png", dpi=150); plt.close()
print(importances.head(10).round(4))
RESULTS["feature_importance_top10"] = importances.head(10).round(4).to_dict()

median_income                 0.4879
ocean_proximity_INLAND        0.1417
longitude                     0.1073
latitude                      0.1022
housing_median_age            0.0509
population                    0.0325
total_rooms                   0.0234
total_bedrooms                0.0230
households                    0.0173
ocean_proximity_NEAR OCEAN    0.0062
dtype: float64


## 14. Análisis de las 10 observaciones con mayor error

In [22]:
err_df = X_test.copy()
err_df["y_real"] = y_test.values
err_df["y_pred"] = best_pred
err_df["residual_abs"] = np.abs(err_df["y_real"] - err_df["y_pred"])
worst10 = err_df.sort_values("residual_abs", ascending=False).head(10)
print(worst10[["median_income","housing_median_age","population","ocean_proximity","y_real","y_pred","residual_abs"]].round(2).to_string())
worst10.to_csv("peores_predicciones.csv", index=False)

resumen_worst = {
    "n_en_tope_500001": int((worst10["y_real"] >= 500000).sum()),
    "media_ingreso_worst10": round(float(worst10["median_income"].mean()), 4),
    "media_ingreso_test": round(float(err_df["median_income"].mean()), 4),
    "ocean_proximity_worst10": worst10["ocean_proximity"].value_counts().to_dict(),
    "error_medio_worst10": round(float(worst10["residual_abs"].mean()), 2),
}
print(resumen_worst)
RESULTS["worst10"] = resumen_worst

       median_income  housing_median_age  population ocean_proximity    y_real     y_pred  residual_abs
12389           3.77                24.0       473.0          INLAND  500001.0  148378.67     351622.33
6688            0.50                28.0       142.0          INLAND  500001.0  168370.66     331630.34
459             1.17                52.0      1349.0        NEAR BAY  500001.0  201525.00     298476.00
10574           1.97                 6.0       125.0       <1H OCEAN  500001.0  203300.36     296700.64
12069           4.24                 6.0       228.0          INLAND  500001.0  203538.33     296462.67
4548            7.58                52.0        55.0       <1H OCEAN   67500.0  359696.07     292196.07
20349           7.30                32.0        63.0      NEAR OCEAN  125000.0  406206.70     281206.70
20325           4.58                21.0       863.0       <1H OCEAN  500001.0  219751.34     280249.66
13015           3.23                11.0      1520.0          IN

### Interpretación de las observaciones con mayor error

Las 10 observaciones peor predichas comparten un patrón muy claro: **6 de las 10 tienen un
valor real de exactamente USD 500.001**, es decir, están en el tope censurado del dataset. El
modelo predice para ellas entre 148.000 y 220.000 dólares, generando errores de casi 300.000
dólares en promedio. Esto no es un fallo del modelo sino una consecuencia directa del
truncamiento del target: las viviendas realmente caras fueron todas registradas con el mismo
valor tope, de modo que el modelo, entrenado mayoritariamente con distritos de precio medio,
no tiene forma de aprender a proyectar por encima de ese techo.

El segundo patrón corresponde a los casos inversos (filas 4548 y 20349): distritos con
ingresos medianos muy altos (7,58 y 7,30) pero valores de vivienda bajos (67.500 y 125.000
dólares). Son distritos con muy poca población (55 y 63 habitantes), es decir, agregaciones
censales atípicamente pequeñas donde la relación ingreso-precio se rompe. El ingreso medio de
las 10 peores observaciones (3,61) es prácticamente igual al del conjunto de test (3,76), lo
que confirma que el error no se explica por el nivel de ingreso sino por la censura del target
y por distritos de tamaño anómalo.

## 15. Selección del mejor modelo

El modelo seleccionado es el **Random Forest**. Sobre el conjunto de test alcanza un
**R² de 0,8175** frente a 0,6251 de Ridge, un **RMSE de 48.901 dólares** frente a 70.094, un
**MAE de 31.683** frente a 51.453 y un **MAPE de 17,7%** frente a 30,1%. La diferencia es
amplia y consistente en las cuatro métricas, no marginal.

**¿Hay sobreajuste?** Sí, y es importante reconocerlo: el Random Forest obtiene un R² de 0,976
en entrenamiento contra 0,8175 en test, una brecha de 0,16 puntos. La búsqueda en grilla
seleccionó `max_depth=None` y `min_samples_split=2`, es decir, árboles completamente
desarrollados, lo que memoriza parcialmente el conjunto de entrenamiento. Sin embargo, el
promedio de 300 árboles sobre muestras bootstrap contiene la varianza: el R² de validación
cruzada (0,818) coincide casi exactamente con el de test (0,8175), lo que confirma que el
desempeño generaliza y que la brecha train-test es la esperada en un ensemble de árboles, no
una señal de que el modelo esté roto. Ridge, en cambio, muestra R² de 0,6695 en train contra
0,6251 en test: bajo sobreajuste, pero también un techo de desempeño claramente inferior,
porque la relación entre ubicación geográfica y precio es fuertemente no lineal y un modelo
lineal no puede capturarla.

**Trade-off desempeño/interpretabilidad:** Ridge entrega coeficientes directamente
interpretables en dólares por unidad de cada predictor, mientras que el Random Forest sólo
entrega importancias relativas. Aun así, la ganancia de 0,19 puntos de R² y la reducción de
más de 21.000 dólares en el RMSE justifican ampliamente la pérdida de interpretabilidad
directa, especialmente porque las importancias del bosque siguen siendo interpretables en
lenguaje de negocio.

**Tiempo de inferencia:** el Random Forest predice las 4.128 observaciones de test en 0,27
segundos frente a 0,0005 de Ridge. En términos absolutos es una diferencia de milisegundos por
predicción, irrelevante para un caso de uso de tasación de propiedades, donde no se requiere
inferencia en tiempo real. El entrenamiento sí es mucho más costoso (814 s contra 1,5 s), pero
es un costo que se paga una sola vez.

## 16. Interpretación de las 5 variables más importantes

1. **`median_income` (0,4879)** — Explica casi la mitad de la capacidad predictiva del modelo.
   En lenguaje de negocio: el poder adquisitivo del barrio determina el precio de la vivienda.
   Coincide con el EDA, donde era la variable más correlacionada con el target (r=0,688), muy
   por encima del resto.
2. **`ocean_proximity_INLAND` (0,1417)** — Estar tierra adentro, lejos de la costa, deprime
   sistemáticamente el valor. Es el efecto "premium costero" de California: el violin plot del
   EDA ya mostraba que los distritos INLAND tienen una distribución de precios claramente
   desplazada hacia abajo.
3. **`longitude` (0,1073)** y 4. **`latitude` (0,1022)** — Juntas suman más de 0,20 de
   importancia y codifican el efecto "ubicación": el eje costa-interior y el eje
   norte-sur (área de la bahía de San Francisco y Los Ángeles frente al valle central). Es
   interesante que en el EDA la latitud tenía una correlación lineal baja (0,144): su efecto
   es no lineal y sólo el modelo de árboles logra capturarlo, lo que explica buena parte de la
   ventaja del Random Forest sobre Ridge.
5. **`housing_median_age` (0,0509)** — La antigüedad del parque habitacional actúa como proxy
   de la consolidación del barrio; barrios antiguos y bien ubicados mantienen valor.

La comparación con el EDA es reveladora: la correlación lineal de Pearson sólo identificaba
correctamente a `median_income`; las variables geográficas parecían irrelevantes en términos
lineales pero resultan ser el segundo bloque explicativo más importante del modelo.

## 17. Conclusiones ejecutivas

**Hallazgos del EDA.** El dataset de California Housing (20.640 distritos censales, 10
variables) resultó estar en buenas condiciones: sólo `total_bedrooms` presentó valores
faltantes, con apenas un 1% de nulos, resuelto mediante imputación por mediana dentro del
pipeline. El análisis de outliers con el método IQR detectó entre un 4% y un 6% de valores
extremos en las variables de tamaño del distrito (`total_rooms`, `population`, `households`),
tratados por winsorización en lugar de eliminación, para no sesgar el modelo hacia distritos
pequeños. La variable objetivo mostró una asimetría de 0,98, bajo el umbral de 1,0, por lo que
se conservó en escala original. El hallazgo más relevante del EDA fue el truncamiento del
target en USD 500.001 y la fuerte multicolinealidad entre las cuatro variables de tamaño del
distrito, con correlaciones de hasta 0,98 entre `total_bedrooms` y `households`.

**Patrones del análisis no supervisado.** El PCA requirió 4 componentes para explicar el 90,7%
de la varianza, y sus cargas confirmaron la estructura detectada en el EDA: PC1 está dominado
por el bloque de tamaño del distrito (hogares, dormitorios, habitaciones y población, todas
con cargas en torno a 0,47), mientras que PC2 captura de forma casi pura la dimensión
geográfica (latitud y longitud, con cargas sobre 0,68). El K-Means alcanzó su mejor Silhouette
Score en K=2 (0,317), y el perfil de clusters mostró que la segmentación natural del dataset
es por **tamaño del distrito** —el cluster 0 promedia 2.246 habitantes y 4.177 habitaciones
frente a 972 y 1.752 del cluster 1— y no por precio, ya que los valores medios de vivienda de
ambos clusters son similares (216.910 contra 203.279 dólares). Es decir, la estructura no
supervisada del dataset no está alineada con la variable objetivo.

**Modelo seleccionado.** El Random Forest (300 árboles, sin límite de profundidad) supera a
Ridge en las cuatro métricas: R²=0,8175 contra 0,6251, RMSE de 48.901 contra 70.094 dólares y
MAPE de 17,7% contra 30,1%. La razón es que la relación entre ubicación geográfica y precio es
marcadamente no lineal, algo que un modelo lineal penalizado no puede representar.

**Top-3 variables.** (1) `median_income`: el poder adquisitivo del barrio es, por lejos, el
principal determinante del precio. (2) `ocean_proximity_INLAND`: la distancia a la costa
genera un descuento sistemático en el valor. (3) Las coordenadas geográficas, que en conjunto
codifican el efecto barrio de los polos urbanos de California.

**Limitaciones.** Primero, los datos provienen del censo de 1990 y no reflejan el mercado
inmobiliario actual, por lo que el modelo no es desplegable para tasación real sin
reentrenamiento. Segundo, el truncamiento del target en 500.001 dólares provoca subestimación
sistemática en el segmento alto: 6 de las 10 peores predicciones son precisamente distritos en
ese tope. Tercero, las variables están agregadas a nivel de distrito censal y no de vivienda
individual, de modo que el modelo predice promedios de barrio y no puede capturar la variación
entre propiedades dentro de un mismo distrito.

**Recomendaciones.** Primero, tratar los distritos censurados como un problema separado, por
ejemplo con un modelo de regresión censurada (Tobit) o un clasificador binario que primero
identifique si un distrito supera el tope y luego aplique un modelo específico a ese segmento.
Segundo, reducir la redundancia del bloque de tamaño construyendo variables derivadas por
hogar —habitaciones por hogar, personas por hogar, proporción de dormitorios— en lugar de usar
los totales crudos, que aportan la misma información cuatro veces.

**Trabajo futuro.** Entrenar un modelo de gradient boosting (XGBoost o LightGBM) incorporando
variables geoespaciales derivadas, como la distancia a los centros urbanos de San Francisco y
Los Ángeles o la densidad de población en un radio dado, para reemplazar la codificación cruda
de latitud y longitud por variables con significado económico directo.

In [23]:
import json
with open("resultados.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print("Análisis completo. Resultados guardados en resultados.json")

Análisis completo. Resultados guardados en resultados.json
